# 01 — Imagining the future increases optimism for related events

Reproduces the Time × Topic × Condition mixed model and its decomposition.

**Expected:** Time×Topic β=7.26, 95%CI[2.93,11.60], p=.001; related Δ=+2.94 (t(150)=2.58, p=.011); unrelated Δ=−4.31 (t(150)=−4.72, p<.001); main effects and 3-way null. Decomposition: unrelated decline ≈59% of the interaction; equivalent across conditions (−3.97 vs −4.66; +1.94 vs +3.95).

In [1]:
import pandas as pd, numpy as np, warnings
from scipy import stats
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')
DATA = '../data'
def load_belief(): return pd.read_excel(f'{DATA}/belief_items.xlsx')
def load_subject(): return pd.read_excel(f'{DATA}/subject_measures.xlsx')
def load_group(): return pd.read_csv(f'{DATA}/group_conversation.csv')
def rep(m,t):
    ci=m.conf_int().loc[t]
    return f'b={m.params[t]:+.3f} 95%CI[{ci[0]:+.3f}, {ci[1]:+.3f}] p={m.pvalues[t]:.4f}'


In [2]:
b=load_belief()
# aggregate to participant x topic x time (mean over items within topic)
agg=b.groupby(['participant','condition','topic'])[['pre','post']].mean().reset_index()
agg['change']=agg.post-agg.pre
L=agg.melt(['participant','condition','topic'],['pre','post'],'time','lik')
L['time_n']=(L.time=='post').astype(int); L['topic_n']=(L.topic=='related').astype(int)
# Interaction: dummy topic + effect-coded condition reproduces the manuscript's beta = related-minus-unrelated change
m=smf.mixedlm('lik ~ time_n*topic_n*C(condition, Sum)', L, groups=L.participant).fit()
print('Time x Topic interaction:', rep(m,'time_n:topic_n'))
# Main effects are null in the manuscript. Their beta magnitudes depend on contrast coding, but the
# tests (p-values) are coding-invariant; the time main effect equals the grand-mean pre->post change.
mf=smf.mixedlm('lik ~ C(time,Sum)*C(topic,Sum)*C(condition,Sum)', L, groups=L.participant).fit()
gm=agg.groupby(['topic','condition']).change.mean().mean()
kt='C(time, Sum)[S.post]'; kc='C(condition, Sum)[S.collaborative]'
k3='C(time, Sum)[S.post]:C(topic, Sum)[S.related]:C(condition, Sum)[S.collaborative]'
# Sum-coded coefficients are 1/2 (two-way factors) and 1/8 (three-way) of the effects reported in the
# manuscript, which are on the interpretable scale (grand-mean change; condition difference;
# difference of differences). Both scalings are printed.
print(f'  time main effect: grand-mean change = {gm:+.2f} (sum-coded b={mf.params[kt]:+.3f}), p = {mf.pvalues[kt]:.2f}')
print(f'  condition main effect: b = {2*mf.params[kc]:+.2f} (sum-coded b={mf.params[kc]:+.3f}), p = {mf.pvalues[kc]:.2f}')
print(f'  three-way interaction: b = {8*mf.params[k3]:+.2f} (sum-coded b={mf.params[k3]:+.3f}), p = {mf.pvalues[k3]:.2f}')


Time x Topic interaction: b=+7.262 95%CI[+2.897, +11.627] p=0.0011
  time main effect: grand-mean change = -0.68 (sum-coded b=-0.342), p = 0.54
  condition main effect: b = -1.90 (sum-coded b=-0.950), p = 0.36
  three-way interaction: b = -2.70 (sum-coded b=-0.338), p = 0.54


In [3]:
# simple effects: related and unrelated pre->post change (paired t across participants)
w=agg.assign(change=lambda x:x.post-x.pre)
for tp in ['related','unrelated']:
    ch=w[w.topic==tp].groupby('participant').change.mean()
    t=stats.ttest_1samp(ch,0); print(f'{tp:9s} Delta={ch.mean():+.2f}  t({len(ch)-1})={t.statistic:.2f} p={t.pvalue:.4f}')


related   Delta=+2.94  t(150)=2.58 p=0.0108
unrelated Delta=-4.31  t(150)=-4.72 p=0.0000


In [4]:
# decomposition: share of the interaction, and equivalence across conditions
rel=w[w.topic=='related'].groupby('participant').change.mean().mean()
unr=w[w.topic=='unrelated'].groupby('participant').change.mean().mean()
print(f'related Delta={rel:+.2f}, unrelated Delta={unr:+.2f}; |unrel|/|rel|={abs(unr)/abs(rel):.2f}x; '
      f'unrelated share of interaction = {abs(unr)/(abs(unr)+abs(rel))*100:.0f}%')
for tp in ['related','unrelated']:
    for cond in ['collaborative','individual']:
        ch=w[(w.topic==tp)&(w.condition==cond)].groupby('participant').change.mean().mean()
        print(f'  {tp:9s} {cond:13s} Delta={ch:+.2f}')


related Delta=+2.94, unrelated Delta=-4.31; |unrel|/|rel|=1.47x; unrelated share of interaction = 59%
  related   collaborative Delta=+1.94
  related   individual    Delta=+3.95
  unrelated collaborative Delta=-3.97
  unrelated individual    Delta=-4.66
